# 🚀 MotionAI Coach – Google Colab Pose Server (GVHMR)

Notebook dựng server trích xuất **pose 3D thật** cho backend (FAQ Q5: được host pipeline AI lên Colab):

```
backend (máy bạn) --AI_REMOTE_URL--> Cloudflare tunnel --> pose_server (cổng 8100) --> GVHMR-Demo API (cổng 8000, GPU)
```

- `pose_server` (của dự án) nhận video, gọi GVHMR-Demo, đọc `hmr4d_results.pt` rồi trả về 24 khớp SMPL.
- Chạy theo kiểu **job bất đồng bộ** (`POST /jobs` rồi hỏi `GET /jobs/{id}`) vì GVHMR mất vài phút / video,
  vượt giới hạn ~100 giây mỗi request của Cloudflare tunnel.
- Đặt `MODEL = "yolo"` ở Bước 2 nếu chỉ muốn chạy nhanh bằng YOLO pose (không cần checkpoint GVHMR).

### ⚠️ Trước tiên: đổi sang GPU
**Runtime → Change runtime type → T4 GPU → Save.**

### Bước 1: Kiểm tra GPU

In [ ]:
!nvidia-smi || echo '⚠️ Đang chạy CPU. Vào Runtime -> Change runtime type -> chọn T4 GPU!'

### Bước 2: Cấu hình & tải mã nguồn dự án
`CKPT_DRIVE_DIR` là thư mục trên Google Drive giữ checkpoint GVHMR để lần sau không phải tải lại.

In [ ]:
import os, secrets

MODEL = "gvhmr"                     # "gvhmr" (pose 3D thật) hoặc "yolo" (nhanh, pose 2D nâng lên 3D)
PROJECT_REPO = "https://github.com/trunghieu714/projectCNWed.git"
GVHMR_REPO = "https://github.com/doantrunghieu08/GVHMR-Demo.git"
GVHMR_DIR = "/content/GVHMR"
CKPT_DRIVE_DIR = "/content/drive/MyDrive/GVHMR_checkpoints"
STATIC_CAM = True                   # điện thoại đặt cố định trên chân máy khi quay
GVHMR_API_KEY = secrets.token_hex(16)   # khóa nội bộ giữa pose_server và GVHMR-Demo
POSE_SERVER_KEY = secrets.token_hex(16) # khóa backend phải gửi kèm (AI_REMOTE_API_KEY)

!pip install -q fastapi uvicorn python-multipart httpx opencv-python-headless scipy pyngrok gdown
if os.path.exists('/content/projectCNWed'):
    !cd /content/projectCNWed && git pull -q
else:
    !git clone -q {PROJECT_REPO} /content/projectCNWed
print('✅ Đã tải mã nguồn dự án')

### Bước 3 (GVHMR): Cài GVHMR-Demo
`requirements.txt` của GVHMR-Demo ghim torch 2.3 / numpy 1.23 / pytorch3d cho Python 3.10, không cài được trên Python
của Colab. Ô này dùng torch có sẵn của Colab, bỏ các dòng ghim đó và **build pytorch3d từ mã nguồn** (~15–20 phút lần
đầu); wheel được lưu vào Drive nên các lần sau chỉ mất vài giây.

GVHMR-Demo chạy trong tiến trình riêng (Bước 5) nên **không cần khởi động lại runtime** sau khi cài.

In [ ]:
import glob, re
if MODEL == "gvhmr":
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(CKPT_DRIVE_DIR, exist_ok=True)

    if not os.path.exists(GVHMR_DIR):
        !git clone -q {GVHMR_REPO} {GVHMR_DIR}

    skip = re.compile(r'^(--extra-index-url|torch==|torchvision==|numpy==|pytorch3d|cython_bbox|chumpy|ipdb|black)')
    with open(f'{GVHMR_DIR}/requirements.txt', encoding='utf-8') as f:
        reqs = [l.strip() for l in f if l.strip() and not l.startswith('#') and not skip.match(l.strip())]
    with open('/content/gvhmr_requirements.txt', 'w') as f:
        f.write('\n'.join(reqs + ['numpy<2']))
    !pip install -q -r /content/gvhmr_requirements.txt
    # chumpy chỉ cần khi đọc SMPL_*.pkl; setup.py của nó cần --no-build-isolation
    !pip install -q --no-build-isolation chumpy || echo '⚠️ Không cài được chumpy (không ảnh hưởng suy luận GVHMR)'
    !cd {GVHMR_DIR} && pip install -q -e . --no-deps

    # pytorch3d: dùng wheel đã build ở lần trước (nếu khớp phiên bản torch/python), không thì build mới
    import torch, sys
    tag = f"torch{torch.__version__.split('+')[0]}_py{sys.version_info.major}{sys.version_info.minor}"
    wheel_dir = f'{CKPT_DRIVE_DIR}/wheels/{tag}'
    if glob.glob(f'{wheel_dir}/pytorch3d-*.whl'):
        !pip install -q {wheel_dir}/pytorch3d-*.whl
    else:
        os.makedirs(wheel_dir, exist_ok=True)
        print('⏳ Đang build pytorch3d (15–20 phút, chỉ lần đầu)...')
        !FORCE_CUDA=1 pip wheel --no-build-isolation --no-deps -w {wheel_dir} "git+https://github.com/facebookresearch/pytorch3d.git@stable"
        !pip install -q {wheel_dir}/pytorch3d-*.whl
    !python -c "import pytorch3d, hmr4d; print('✅ pytorch3d', pytorch3d.__version__, '+ hmr4d OK')"
else:
    print('Bỏ qua (MODEL = yolo)')

### Bước 4 (GVHMR): Checkpoint
Cấu trúc cần có trong `MyDrive/GVHMR_checkpoints/` (ô dưới sẽ liên kết thư mục này vào `GVHMR/inputs/checkpoints`):

```
body_models/smplx/SMPLX_NEUTRAL.npz   <- BẮT BUỘC tự tải: đăng ký https://smpl-x.is.tue.mpg.de/
body_models/smpl/SMPL_NEUTRAL.pkl     <- tự tải: đăng ký https://smpl.is.tue.mpg.de/
gvhmr/gvhmr_siga24_release.ckpt       ┐
hmr2/epoch=10-step=25000.ckpt         │ ô dưới tự tải từ thư mục Google Drive chính thức của GVHMR
vitpose/vitpose-h-multi-coco.pth      │ (tải về nghĩa là bạn đồng ý license của từng mô hình)
yolo/yolov8x.pt                       ┘
```
SMPL/SMPL-X có license riêng nên không được phân phối lại – mỗi người phải tự đăng ký và tải, rồi upload lên Drive.
Nếu `gdown` báo hết lượt tải (quota), mở link trong log, tải thủ công và upload vào đúng thư mục.

In [ ]:
if MODEL == "gvhmr":
    OFFICIAL_FOLDER = "https://drive.google.com/drive/folders/1eebJ13FUEXrKBawHpJroW0sNSxLjh9xD"
    needed = {
        "body_models/smplx/SMPLX_NEUTRAL.npz": False,   # False = phải tự tải (license SMPL-X)
        "body_models/smpl/SMPL_NEUTRAL.pkl": False,
        "gvhmr/gvhmr_siga24_release.ckpt": True,
        "hmr2/epoch=10-step=25000.ckpt": True,
        "vitpose/vitpose-h-multi-coco.pth": True,
        "yolo/yolov8x.pt": True,
    }
    missing = lambda auto: [p for p, a in needed.items() if a == auto and not os.path.exists(f'{CKPT_DRIVE_DIR}/{p}')]
    if missing(True):
        print('⏳ Tải checkpoint GVHMR từ Drive chính thức:', missing(True))
        !gdown --folder -q {OFFICIAL_FOLDER} -O /content/gvhmr_ckpt_dl --remaining-ok
        for p in missing(True):
            src = glob.glob(f'/content/gvhmr_ckpt_dl/**/{os.path.basename(p)}', recursive=True)
            if src:
                os.makedirs(os.path.dirname(f'{CKPT_DRIVE_DIR}/{p}'), exist_ok=True)
                !cp "{src[0]}" "{CKPT_DRIVE_DIR}/{p}"

    !mkdir -p {GVHMR_DIR}/inputs && rm -rf {GVHMR_DIR}/inputs/checkpoints && ln -s {CKPT_DRIVE_DIR} {GVHMR_DIR}/inputs/checkpoints
    print('\n'.join(('✅ ' if os.path.exists(f'{CKPT_DRIVE_DIR}/{p}') else '❌ THIẾU ') + p for p in needed))
    if missing(False) or missing(True):
        print('\n⚠️ Còn thiếu file – upload vào', CKPT_DRIVE_DIR, 'rồi chạy lại ô này trước khi sang Bước 5.')
else:
    print('Bỏ qua (MODEL = yolo)')

### Bước 5 (GVHMR): Khởi động GVHMR-Demo API (cổng 8000, chỉ trong Colab)
Mô hình nạp vào GPU mất khoảng 1–3 phút; ô này đợi tới khi log báo `Model loaded successfully`.

In [ ]:
import subprocess, time
if MODEL == "gvhmr":
    !fuser -k 8000/tcp 2>/dev/null || true
    gvhmr_log = open('/content/gvhmr.log', 'w')
    gvhmr_proc = subprocess.Popen(
        ['python', '-m', 'uvicorn', 'app.main:app', '--host', '127.0.0.1', '--port', '8000'],
        cwd=GVHMR_DIR, env={**os.environ, 'API_KEY': GVHMR_API_KEY, 'PYTHONPATH': GVHMR_DIR},
        stdout=gvhmr_log, stderr=subprocess.STDOUT)
    for _ in range(300):
        time.sleep(2)
        log = open('/content/gvhmr.log', errors='ignore').read()
        if 'Model loaded successfully' in log or gvhmr_proc.poll() is not None or 'Không tìm thấy checkpoint' in log:
            break
    print(log[-3000:])
    print('✅ GVHMR-Demo đã sẵn sàng' if 'Model loaded successfully' in log else '❌ GVHMR chưa nạp được mô hình – xem log ở trên')
else:
    print('Bỏ qua (MODEL = yolo)')

### Bước 6: Khởi động pose_server của dự án (cổng 8100)

In [ ]:
import httpx
!fuser -k 8100/tcp 2>/dev/null || true
env = {**os.environ, 'PYTHONPATH': '/content/projectCNWed/backend', 'POSE_SERVER_KEY': POSE_SERVER_KEY}
if MODEL == "gvhmr":
    env.update(POSE_BACKEND='gvhmr_api', GVHMR_URL='http://127.0.0.1:8000', GVHMR_API_KEY=GVHMR_API_KEY,
               GVHMR_DIR=GVHMR_DIR, GVHMR_STATIC_CAM='1' if STATIC_CAM else '0', POSE_TIMEOUT='1800')
else:
    !pip install -q ultralytics
    env.update(POSE_BACKEND='yolo', YOLO_POSE_WEIGHTS='yolo11x-pose.pt')
pose_log = open('/content/pose_server.log', 'w')
pose_proc = subprocess.Popen(['python', '-m', 'ai_engine.remote_server.pose_server'], cwd='/content/projectCNWed/backend',
                             env=env, stdout=pose_log, stderr=subprocess.STDOUT)
time.sleep(4)
try:
    print('✅ pose_server online:', httpx.get('http://127.0.0.1:8100/', timeout=5).json())
except Exception as e:
    print('❌ Lỗi:', e); print(open('/content/pose_server.log').read()[-2000:])

### Bước 7: Chạy thử với video mẫu (khuyên làm trước khi demo)
Gửi `docs/example_video/tennis.mp4` của GVHMR-Demo qua đúng giao thức job mà backend sẽ dùng.

In [ ]:
sample = f'{GVHMR_DIR}/docs/example_video/tennis.mp4'
if not os.path.exists(sample):
    !wget -q -O /content/tennis.mp4 https://raw.githubusercontent.com/doantrunghieu08/GVHMR-Demo/main/docs/example_video/tennis.mp4
    sample = '/content/tennis.mp4'
hdr = {'X-API-Key': POSE_SERVER_KEY}
with open(sample, 'rb') as f:
    job_id = httpx.post('http://127.0.0.1:8100/jobs', headers=hdr, files={'file': ('tennis.mp4', f, 'video/mp4')}, timeout=120).json()['job_id']
t0 = time.time()
while True:
    job = httpx.get(f'http://127.0.0.1:8100/jobs/{job_id}', headers=hdr, timeout=30).json()
    print(f"\r[{time.time()-t0:5.0f}s] {job['status']}: {job.get('progress')}", end='')
    if job['status'] in ('COMPLETED', 'FAILED'):
        break
    time.sleep(5)
print()
if job['status'] == 'COMPLETED':
    r = job['result']
    print(f"✅ {len(r['joints_3d'])} frame x {len(r['joints_3d'][0])} khớp, nguồn {r['source']}, {r['seconds']}s")
else:
    print('❌', job.get('error'))
    for log in ('/content/pose_server.log', '/content/gvhmr.log'):
        if os.path.exists(log):
            print(f'--- {log}'); print(open(log, errors='ignore').read()[-2000:])

### Bước 8: Mở tunnel ra Internet (Cloudflare) và lấy cấu hình cho backend

In [ ]:
!which cloudflared > /dev/null || (wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb > /dev/null)
!killall cloudflared 2>/dev/null || true
logfile = '/content/cloudflared.log'
if os.path.exists(logfile):
    os.remove(logfile)
!nohup cloudflared tunnel --url http://127.0.0.1:8100 --logfile /content/cloudflared.log > /dev/null 2>&1 &

tunnel_url = None
for _ in range(30):
    time.sleep(1)
    if os.path.exists(logfile):
        m = re.findall(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', open(logfile, errors='ignore').read())
        if m:
            tunnel_url = m[-1]
            break

def print_env(url):
    print('=' * 75)
    print('📋 DÁN VÀO backend/.env TRÊN MÁY BẠN rồi khởi động lại backend (và Celery worker nếu có):')
    print(f'AI_REMOTE_URL={url}')
    print(f'AI_REMOTE_API_KEY={POSE_SERVER_KEY}')
    print('AI_ESTIMATOR_TYPE=remote,precomputed,mock')
    print('=' * 75)

if tunnel_url:
    print_env(tunnel_url)
else:
    print('⚠️ Chưa lấy được URL Cloudflare – dùng Ngrok ở ô dưới. Log:')
    print(open(logfile, errors='ignore').read() if os.path.exists(logfile) else '')

### (Tùy chọn) Bước 9: Ngrok nếu mạng chặn Cloudflare

In [ ]:
from pyngrok import ngrok
NGROK_TOKEN = ''  # token miễn phí: https://dashboard.ngrok.com/get-started/your-authtoken
if NGROK_TOKEN.strip():
    ngrok.kill(); ngrok.set_auth_token(NGROK_TOKEN.strip())
    print_env(ngrok.connect(8100).public_url)
else:
    print('Bỏ qua nếu Bước 8 đã in ra link Cloudflare.')

### Khi gặp lỗi
- Log GVHMR: `!tail -50 /content/gvhmr.log` · Log pose_server: `!tail -50 /content/pose_server.log`
- Colab ngắt kết nối sau ~90 phút không tương tác / 12 giờ: chạy lại từ Bước 2 (checkpoint & wheel pytorch3d đã nằm trên Drive). URL tunnel đổi mỗi lần → cập nhật lại `.env`.
- Backend vẫn chấm được nếu Colab tắt: `AI_ESTIMATOR_TYPE` là chuỗi dự phòng, nguồn pose thực tế được ghi trong kết quả chấm.